# Teaching a small model to write the briefings (stage 1: supervised fine-tuning)

This notebook takes a small, free, open model and teaches it to write briefings the way Claude does, using the examples you packaged with `training/package_sft.py`. When it finishes you will have one file (about 2 GB) that you can run on your own Mac, with nothing sent anywhere.

There are only four ideas in the whole thing:

1. **A small open model.** A "model" is a very large table of numbers (called **weights**) that turns text in into text out. Big ones like Claude have hundreds of billions of numbers; the one we use here has about three billion, small enough for a free Google GPU and for your laptop. It already knows English and JSON. It does not know your briefing rules.

2. **Your examples.** Each example is a packet (the group, the reader's ledger, the news event) plus the briefing Claude wrote for it. The exact number is read from the two files you upload; expect a few hundred. That is small for training a model from scratch and plenty for nudging one that already speaks the language.

3. **Teaching by nudging.** We show the model a packet, let it guess the briefing, compare the guess with Claude's, and move the weights a tiny step in the direction that would have made Claude's answer more likely. Then the next example. Then the next. Over a few passes through the data the guesses get close.

4. **The one number to watch.** Every step produces a **loss**: a single number saying how surprised the model was by Claude's answer. Lower is better. We measure it on the examples we train on (train loss) and, separately, on a few dozen examples we hold back and never train on (validation loss). If both fall, the model is learning. If train loss keeps falling but validation loss starts rising, the model has begun memorising instead of learning, and we stop.

Run the cells top to bottom. Each cell has a short note above it saying what it does and what you should see. The whole thing should take about 90 minutes on Colab's free tier; the long parts are the training itself and the final conversion.

**Technical summary.** Base model `Qwen2.5-3B-Instruct`, loaded in 4-bit. LoRA adapter (r=16, alpha=32, dropout 0.05) on all attention and MLP projections. Loss on the assistant turn only. Three epochs, effective batch size 8, evaluated on a held-out 10% split stratified by reader profile. Output is checked against the data generator's hard rules and exported to GGUF for local serving.

**Status.** This notebook has not been run to completion yet; it carries no stored outputs and no result is claimed. The notes below are written for a reader new to fine-tuning.


## 1. Check a GPU is attached

Training needs a **GPU** (a graphics card, which is very good at the kind of arithmetic training is made of). Colab gives you one for free, but you have to ask for it: in the menu go to **Runtime → Change runtime type**, choose **T4 GPU** under "Hardware accelerator", and save. This cell checks it is there and prints how much memory it has. You should see a table with "Tesla T4" and about 15 GB (15360 MiB). If you see an error instead, attach the GPU as above and run this cell again.

In [ ]:
import shutil, subprocess

if shutil.which("nvidia-smi") is None:
    raise SystemExit(
        "No GPU is attached. In the menu: Runtime -> Change runtime type -> T4 GPU -> Save. Then run this cell again."
    )
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
mem = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True
).stdout.strip()
print("GPU:", mem)
total_mib = int(mem.split(",")[1].strip().split()[0])
if total_mib < 14000:
    print("WARNING: this GPU has less than 14 GB. The notebook is sized for a 15 GB T4; consider the 1.5B model (see the model cell).")
else:
    print("OK: enough memory for the 3B model at the sequence lengths this data needs.")

## 2. Install the training tools

This installs the libraries that do the work. **unsloth** is a package that makes training on a single small GPU about twice as fast and use about half the memory; it does that by loading the model in **4-bit**, which means storing each of the model's numbers with 4 bits of precision instead of 16 (a bit less exact, four times smaller, and for our purposes indistinguishable). This is called **quantisation**: storing numbers more coarsely to save space.

The versions are pinned to the release the notebook was written against (a monthly release series, so security patches within it still install). It takes two to four minutes and prints a lot; the only lines that matter are the version table at the end. If it ends with an error, skip to the fallback cell below.

In [ ]:
import subprocess, sys, importlib

USE_UNSLOTH = True  # the fallback cell sets this to False

def pip(*args):
    cmd = [sys.executable, "-m", "pip", "install", "--quiet", *args]
    print("$", " ".join(cmd))
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print(result.stdout[-3000:], result.stderr[-3000:])
        raise RuntimeError("pip failed -- run the fallback cell below instead")

# unsloth pins its own tested versions of trl / peft / transformers / accelerate /
# bitsandbytes / datasets, so we pin unsloth to a release series and let it choose.
pip("--upgrade", "unsloth==2025.9.*", "unsloth_zoo==2025.9.*")
pip("matplotlib")

print()
print("installed versions:")
for name in ["unsloth", "trl", "peft", "transformers", "datasets", "accelerate", "bitsandbytes", "torch"]:
    try:
        mod = importlib.import_module(name)
        print(f"  {name:<14} {getattr(mod, '__version__', '?')}")
    except Exception as exc:  # noqa: BLE001
        print(f"  {name:<14} NOT IMPORTABLE ({exc})")

### Fallback install (only if the cell above failed)

If unsloth would not install, remove the `#` at the start of every line in this cell and run it. It installs the plain, slower libraries instead (training will take roughly twice as long, still within a Colab session). Every later cell checks the `USE_UNSLOTH` switch and takes the right path, so nothing else changes for you. If the cell above worked, leave this one alone.

In [ ]:
# import subprocess, sys
# USE_UNSLOTH = False
# subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--upgrade",
#                 "transformers>=4.55,<4.60", "peft>=0.17,<0.19", "trl>=0.21,<0.25",
#                 "bitsandbytes>=0.46", "accelerate>=1.9", "datasets>=3.6", "matplotlib"], check=True)
# print("fallback libraries installed; USE_UNSLOTH =", USE_UNSLOTH)

## 3. Upload the two data files

When you run this cell a **Choose Files** button appears. Click it and select both `sft_train.jsonl` and `sft_val.jsonl` from `training/data/` on your Mac (you can select both at once). The train file is what the model learns from; the val file is the held-back set we only measure on.

The cell also connects your Google Drive (a pop-up will ask you to allow it). That matters because Colab's free tier can disconnect without warning and wipe its disk: with Drive connected, the notebook saves its progress to Drive after every pass over the data and the finished model file goes there too, so a disconnect costs you at most one pass. If you would rather not connect Drive, set `USE_DRIVE = False`; everything still works, you just lose the safety net.

Expect to see the number of examples in each file and a breakdown by reader profile.

In [ ]:
import json, os, collections

USE_DRIVE = True
TRAIN_FILE, VAL_FILE = "sft_train.jsonl", "sft_val.jsonl"

WORK_DIR = "/content/news-briefer"
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK_DIR = "/content/drive/MyDrive/news-briefer"
os.makedirs(WORK_DIR, exist_ok=True)
print("working folder:", WORK_DIR)

missing = [f for f in (TRAIN_FILE, VAL_FILE) if not os.path.exists(f)]
if missing:
    from google.colab import files
    print("Please choose:", ", ".join(missing))
    uploaded = files.upload()
    still_missing = [f for f in (TRAIN_FILE, VAL_FILE) if f not in uploaded and not os.path.exists(f)]
    if still_missing:
        raise SystemExit(f"Still missing {still_missing}. Run this cell again and pick both files.")

def read_jsonl(path):
    with open(path, encoding="utf-8") as fh:
        return [json.loads(line) for line in fh if line.strip()]

train_rows = read_jsonl(TRAIN_FILE)
val_rows = read_jsonl(VAL_FILE)
for name, rows in (("train", train_rows), ("val", val_rows)):
    profiles = collections.Counter(r["meta"].get("profile", "?") for r in rows)
    print(f"{name}: {len(rows)} examples  ", dict(sorted(profiles.items())))

example = train_rows[0]
assert [m["role"] for m in example["messages"]] == ["system", "user", "assistant"], "unexpected row shape"
print()
print("One example, so you can see the three parts:")
print("  system    (the instructions + output format):", len(example["messages"][0]["content"]), "characters")
print("  user      (the packet):                      ", len(example["messages"][1]["content"]), "characters")
print("  assistant (Claude's answer):                 ", len(example["messages"][2]["content"]), "characters")
print()
print(example["messages"][2]["content"][:400], "...")

## 4. Load the base model

This downloads the model we start from: **Qwen2.5-3B-Instruct**, an open model from Alibaba with about three billion **weights** (the numbers that make up the model), already trained to follow instructions, in a **4-bit** copy (each number stored in 4 bits, so the whole model is under 2 GB instead of 6). It fits a T4 with room to spare for training. The download takes one to three minutes.

If you would like to try the newer, slightly larger **Qwen3-4B** instead, change `MODEL_NAME` to `unsloth/Qwen3-4B-Instruct-2507` (about 35% slower, a bit better at instructions). If training turns out too slow, `unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit` is half the size and about twice as fast.

The cell also prints the model's **chat template**: the exact way it expects a conversation to be laid out as text, with special markers like `<|im_start|>system`. Everything you upload is rendered through this template before the model sees it, so it is worth one look. You should see the template, then one of your examples rendered through it (cut short).

In [ ]:
import torch

MODEL_NAME = "unsloth/Qwen2.5-3B-Instruct-bnb-4bit"
# Alternatives: "unsloth/Qwen3-4B-Instruct-2507"  (larger, slower)
#               "unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit"  (smaller, faster)
MODEL_CEILING = 6144  # an upper bound for now; the exact training length is measured from the data two cells down

if USE_UNSLOTH:
    from unsloth import FastLanguageModel
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_NAME,
        max_seq_length=MODEL_CEILING,
        dtype=None,          # unsloth picks float16 on a T4
        load_in_4bit=True,
    )
else:
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    bnb = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True,
    )
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, quantization_config=bnb, device_map={"": 0})

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("loaded", MODEL_NAME)
print("GPU memory used by the model alone: %.1f GB" % (torch.cuda.memory_allocated() / 1e9))
print()
print("=== the chat template (what the model actually reads) ===")
print(tokenizer.chat_template)
print()
print("=== one of your examples rendered through it (first 600 characters, then the last 300) ===")
rendered = tokenizer.apply_chat_template(train_rows[0]["messages"], tokenize=False)
print(rendered[:600])
print("   ...")
print(rendered[-300:])

## 5. Attach the part we will actually train (LoRA)

We do not change the model's three billion numbers directly; that would need far more memory than a T4 has, and with a few hundred examples it would also be easy to wreck what the model already knows. Instead we bolt on a small **adapter** (a few tens of millions of extra numbers, about 1% of the model) and train only that; this technique is called **LoRA**. The original weights stay frozen and the adapter learns the adjustment. At the end we can either keep the adapter as a small separate file or fold it back into the model.

The three settings: `r=16` is the adapter's size, `alpha=32` is how strongly it is allowed to pull, and `dropout=0.05` randomly switches off 5% of it during each step so it cannot lean on any one connection (a light guard against memorising). You should see a line saying how many numbers are trainable, around 1% of the total.

In [ ]:
LORA_R, LORA_ALPHA, LORA_DROPOUT = 16, 32, 0.05
TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

if USE_UNSLOTH:
    model = FastLanguageModel.get_peft_model(
        model,
        r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
        target_modules=TARGET_MODULES, bias="none",
        use_gradient_checkpointing="unsloth",   # trades a little speed for a lot of memory; needed at these lengths
        random_state=3407,
    )
else:
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
    model = get_peft_model(model, LoraConfig(
        r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
        target_modules=TARGET_MODULES, bias="none", task_type="CAUSAL_LM",
    ))
    model.gradient_checkpointing_enable()

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"trainable numbers: {trainable/1e6:.1f} million of {total/1e9:.2f} billion ({100*trainable/total:.2f}%)")

## 6. Measure how long the examples are

Models do not read characters; they read **tokens**, chunks of roughly four characters (a short word, or part of a long one). Every example has to fit inside a fixed window of tokens, so this cell renders each example through the chat template, counts its tokens, and picks the window (`max_seq_length`) from the longest one, rounded up. Nothing gets cut off.

Expect the longest example to be a little over 4,000 tokens, because the briefing instructions alone are about 2,800 tokens and they are in every example. That is long for a T4, which is why we train one example at a time with **gradient checkpointing** (recomputing some intermediate results instead of storing them, which roughly halves the memory needed). At 4,352 tokens, 4-bit weights plus the adapter plus the working memory come to about 8 to 10 GB of the T4's 15, so it fits. If the cell warns that something would be truncated, that is the thing to fix before training; it should not happen with the packaged files.

In [ ]:
import statistics
from datasets import Dataset

def token_len(messages):
    return len(tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=False))

train_lens = [token_len(r["messages"]) for r in train_rows]
val_lens = [token_len(r["messages"]) for r in val_rows]
all_lens = sorted(train_lens + val_lens)
p95 = all_lens[min(len(all_lens) - 1, round(0.95 * (len(all_lens) - 1)))]
longest = all_lens[-1]
system_tokens = len(tokenizer(train_rows[0]["messages"][0]["content"])["input_ids"])
answer_lens = [len(tokenizer(r["messages"][2]["content"])["input_ids"]) for r in train_rows]

print(f"examples: {len(all_lens)}   tokens per example: mean {round(statistics.mean(all_lens))}, 95th percentile {p95}, longest {longest}")
print(f"the system message alone is {system_tokens} tokens; Claude's answers average {round(statistics.mean(answer_lens))} tokens (longest {max(answer_lens)})")

max_seq_length = min(MODEL_CEILING, ((longest + 96) // 256 + 1) * 256)  # a little headroom, rounded up to a multiple of 256
truncated = sum(1 for n in all_lens if n > max_seq_length)
if truncated:
    print(f"WARNING: {truncated} examples are longer than max_seq_length={max_seq_length} and would be cut off. "
          "Raise MODEL_CEILING in the model cell and re-run from there.")
else:
    print(f"max_seq_length = {max_seq_length}: every example fits, nothing will be truncated.")
tokens_per_epoch = sum(train_lens)
print(f"one pass over the training set is {tokens_per_epoch/1e6:.2f} million tokens")

# Build the datasets. With unsloth we hand over the rendered text and mask the prompt later;
# without it we hand over prompt/completion pairs and TRL masks the prompt itself.
if USE_UNSLOTH:
    train_ds = Dataset.from_list([{"text": tokenizer.apply_chat_template(r["messages"], tokenize=False)} for r in train_rows])
    val_ds = Dataset.from_list([{"text": tokenizer.apply_chat_template(r["messages"], tokenize=False)} for r in val_rows])
else:
    train_ds = Dataset.from_list([{"prompt": r["messages"][:2], "completion": r["messages"][2:]} for r in train_rows])
    val_ds = Dataset.from_list([{"prompt": r["messages"][:2], "completion": r["messages"][2:]} for r in val_rows])
print("datasets ready:", len(train_ds), "train,", len(val_ds), "val")

## 7. Train

This is the long cell: expect roughly an hour on a T4 for three passes. Words you will see in its output:

- An **epoch** is one full pass over the training examples. We do three. Fewer and the model has not settled; many more and it starts memorising.
- The **loss** is the model's surprise at Claude's answer, averaged over the tokens it is being taught. It starts somewhere around 1.0 to 1.5 and should fall quickly in the first few dozen steps, then more slowly. **`eval_loss`** is the same number measured on the held-back validation examples, printed every 25 steps. A healthy run has both numbers falling together and flattening out; a run that is memorising shows train loss still dropping while `eval_loss` turns upward. The plot in the next cell makes this easy to see.
- A **step** is one adjustment of the weights. We look at 8 examples per step (one at a time, adding up what we learn from each, which is what "gradient accumulation" means), so an epoch is about the training-set size divided by 8 steps.

Two details worth knowing. First, we only teach the model on the **assistant** part of each example, not on the instructions or the packet: those are given to it at serve time, so there is nothing to learn from predicting them, and skipping them keeps the loss focused on the briefing. Second, a checkpoint of the adapter is saved to your working folder at the end of every epoch, and if the cell finds one there when it starts it carries on from it, so a Colab disconnect costs at most one epoch: reconnect, run the cells from the top (the upload cell will ask for the files again), and this cell resumes.

After the first five steps the cell prints a projected total time. If that is well over 80 minutes, stop the cell, set `EPOCHS = 2`, or switch to the 1.5B model in the model cell, and run again from there.

In [ ]:
import os, time
from transformers import TrainerCallback
from transformers.trainer_utils import get_last_checkpoint
from trl import SFTConfig, SFTTrainer

EPOCHS = 3
LEARNING_RATE = 2e-4
EVAL_EVERY = 25
OUTPUT_DIR = os.path.join(WORK_DIR, "checkpoints")
os.makedirs(OUTPUT_DIR, exist_ok=True)

class LossLog(TrainerCallback):
    # Keeps every logged number in a plain list so the plot cell can use it,
    # and prints a time projection once the first few steps are done.
    def __init__(self):
        self.rows, self.t0 = [], None
    def on_train_begin(self, args, state, control, **kw):
        self.t0 = time.time()
    def on_log(self, args, state, control, logs=None, **kw):
        if logs:
            self.rows.append(dict(logs, step=state.global_step))
    def on_step_end(self, args, state, control, **kw):
        if state.global_step == 5 and self.t0:
            per_step = (time.time() - self.t0) / 5
            print(f"\n[projection] {per_step:.0f} s per step x {state.max_steps} steps = about {per_step*state.max_steps/60:.0f} minutes for training\n")

loss_log = LossLog()

config_kwargs = dict(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    per_device_eval_batch_size=1,
    eval_strategy="steps",
    eval_steps=EVAL_EVERY,
    logging_steps=5,
    save_strategy="epoch",
    save_total_limit=2,
    fp16=not torch.cuda.is_bf16_supported(),   # the T4 has no bf16, so fp16
    bf16=torch.cuda.is_bf16_supported(),
    optim="adamw_8bit",
    weight_decay=0.01,
    seed=3407,
    report_to="none",
)
if USE_UNSLOTH:
    config_kwargs["dataset_text_field"] = "text"     # we hand over rendered text; the prompt is masked below
else:
    config_kwargs["completion_only_loss"] = True     # prompt/completion pairs; TRL masks the prompt itself
try:
    sft_config = SFTConfig(max_length=max_seq_length, **config_kwargs)
except TypeError:  # older TRL spelling
    sft_config = SFTConfig(max_seq_length=max_seq_length, **config_kwargs)

trainer_kwargs = dict(model=model, train_dataset=train_ds, eval_dataset=val_ds, args=sft_config, callbacks=[loss_log])
try:
    trainer = SFTTrainer(processing_class=tokenizer, **trainer_kwargs)
except TypeError:  # older TRL spelling
    trainer = SFTTrainer(tokenizer=tokenizer, **trainer_kwargs)

if USE_UNSLOTH:
    # Teach on the assistant turn only. These two strings are how Qwen's template opens
    # the user and assistant turns; everything before the response marker is ignored by the loss.
    from unsloth.chat_templates import train_on_responses_only
    trainer = train_on_responses_only(
        trainer,
        instruction_part="<|im_start|>user\n",
        response_part="<|im_start|>assistant\n",
    )

last = get_last_checkpoint(OUTPUT_DIR) if os.listdir(OUTPUT_DIR) else None
if last:
    print("Found a checkpoint from an earlier attempt, resuming from:", last)
print("GPU memory before training: %.1f GB" % (torch.cuda.memory_allocated() / 1e9))
result = trainer.train(resume_from_checkpoint=last)
print()
print("done. peak GPU memory: %.1f GB of %.1f GB" % (torch.cuda.max_memory_allocated() / 1e9, torch.cuda.get_device_properties(0).total_memory / 1e9))
print("training time: %.0f minutes" % (result.metrics.get("train_runtime", 0) / 60))

## 8. Look at the loss curve

This draws the two loss numbers against the step count: train loss as a thin line, validation loss as dots. How to read it:

- **Both falling, then flattening**: the model learned the pattern and stopped improving. That is what you want.
- **Train falling, validation flat or rising** (the lines part ways and the dots go up): the model is memorising the training examples instead of learning the rule; this is called **overfitting**. If it starts after epoch 2, the epoch-2 checkpoint in your working folder is the better model, and the adapter cell below tells you how to use it.
- **Neither falling much**: something is wrong with the data or the learning rate; do not carry on to the export cells.

The dashed vertical lines mark the end of each epoch.

In [ ]:
import matplotlib.pyplot as plt

rows = trainer.state.log_history or loss_log.rows
train_pts = [(r["step"], r["loss"]) for r in rows if "loss" in r]
val_pts = [(r["step"], r["eval_loss"]) for r in rows if "eval_loss" in r]

fig, ax = plt.subplots(figsize=(8, 4.5))
if train_pts:
    ax.plot(*zip(*train_pts), color="#4C72B0", linewidth=1.5, label="train loss (examples it learns from)")
if val_pts:
    ax.plot(*zip(*val_pts), color="#DD8452", marker="o", linewidth=1.5, label="validation loss (held-back examples)")
steps_per_epoch = trainer.state.max_steps / EPOCHS if trainer.state.max_steps else None
if steps_per_epoch:
    for e in range(1, EPOCHS):
        ax.axvline(e * steps_per_epoch, color="#999999", linestyle="--", linewidth=1)
ax.set_xlabel("step")
ax.set_ylabel("loss (lower is better)")
ax.set_title("Did it learn, or memorise?")
ax.grid(alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

if val_pts:
    best_step, best_val = min(val_pts, key=lambda p: p[1])
    print(f"lowest validation loss {best_val:.3f} at step {best_step} (last measured {val_pts[-1][1]:.3f} at step {val_pts[-1][0]})")
    if val_pts[-1][1] > best_val * 1.05:
        print("Validation loss rose more than 5% from its best: the model started memorising. Prefer the checkpoint nearest the best step.")
    else:
        print("Validation loss did not climb back up: no sign of memorising.")

## 9. Try it: two held-back packets, the student next to the teacher

This is your first look at what training bought. The cell picks two validation examples the model has never seen, one for a complete beginner and one for a reader who is expert in one corner of the group, and has the trained model write the briefing for each. For each it prints the key facts of the packet (who the reader is, what they already know, the event), then the model's briefing next to Claude's.

Underneath each pair it runs four of the data generator's hard rules plus a JSON-shape check to accept an example: the answer must be valid JSON with every required field, contain no question mark, use no number that is not in the event, define no term the ledger says the reader already knows, and list as "explained" only terms it also lists as "used". Every one of Claude's answers passed these by construction; the interesting column is the model's. It also prints a short depth readout for both (reading grade level, average sentence length, how dense the domain terms are, and how many definitions per 100 words), because "pitched at the right level" is a matter of degree the pass/fail rules do not capture.

Generation takes about 20 to 40 seconds per briefing on a T4.

In [ ]:
import json, re, textwrap

# ---- hard rules, reimplemented minimally (four of training/generate_sft.py's, plus a JSON-shape check) ----
KNOWN_STATES = ("provisional", "familiar", "explained", "confirmed")
REQUIRED_KEYS = ("briefing", "topic", "terms_used", "explained_terms", "subdomains", "reasoning")
_NUMBER = re.compile(r"\d{1,3}(?:,\d{3})+(?:\.\d+)?|\d+(?:\.\d+)?")
_ORDINAL_TAIL = re.compile(r"^(st|nd|rd|th)\b", re.IGNORECASE)

def norm_term(term):
    cleaned = str(term).strip().lower().replace("-", " ").replace("_", " ")
    return " ".join(cleaned.split())

def figure_key(number):
    digits = re.sub(r"[^0-9]", "", number).lstrip("0").rstrip("0")
    return digits or "0"

def numbers_in(text):
    out = []
    for m in _NUMBER.finditer(text or ""):
        num = m.group(0)
        if re.fullmatch(r"(19|20)\d\d", num):
            continue
        if _ORDINAL_TAIL.match(text[m.end(): m.end() + 2]):
            continue
        out.append((num, figure_key(num)))
    return out

def parse_answer(text):
    # The first {...} object in the reply; tolerates a stray sentence or a code fence around it.
    if not text:
        return None
    start = text.find("{")
    while start != -1:
        depth, in_str, esc = 0, False, False
        for i in range(start, len(text)):
            ch = text[i]
            if in_str:
                if esc: esc = False
                elif ch == "\\": esc = True
                elif ch == '"': in_str = False
                continue
            if ch == '"': in_str = True
            elif ch == "{": depth += 1
            elif ch == "}":
                depth -= 1
                if depth == 0:
                    try:
                        obj = json.loads(text[start:i + 1])
                        return obj if isinstance(obj, dict) else None
                    except json.JSONDecodeError:
                        break
        start = text.find("{", start + 1)
    return None

def hard_rule_checks(context, answer_text):
    # `context` is the packet's `context` object; `answer_text` is the raw model reply.
    out = parse_answer(answer_text)
    parsed_ok = out is not None and all(k in out for k in REQUIRED_KEYS)
    out = out or {}
    briefing = str(out.get("briefing") or "")
    event = (context.get("events") or [{}])[0]
    source = f"{event.get('headline') or ''} {event.get('detail') or ''}"
    source_keys = {k for _, k in numbers_in(source)}
    unsupported = [n for n, k in numbers_in(briefing) if k not in source_keys]
    known = {norm_term(t) for s in KNOWN_STATES for t in (context.get("concept_ledger") or {}).get(s, [])}
    explained = [norm_term(t) for t in out.get("explained_terms") or [] if isinstance(t, str)]
    used = {norm_term(t) for t in out.get("terms_used") or [] if isinstance(t, str)}
    checks = {
        "json_parses_with_required_keys": parsed_ok,
        "no_question_mark": parsed_ok and "?" not in briefing,
        "numbers_supported": parsed_ok and not unsupported,
        "no_regloss_of_known_terms": parsed_ok and not any(t in known for t in explained),
        "explained_subset_of_used": parsed_ok and all(t in used for t in explained),
    }
    checks["passed"] = all(checks.values())
    checks["details"] = {
        "unsupported_numbers": unsupported,
        "reglossed": sorted({t for t in explained if t in known}),
        "explained_not_in_used": sorted({t for t in explained if t not in used}),
    }
    return checks

# ---- depth readout: how the briefing is pitched, as numbers rather than a verdict ----
_WORD_RE = re.compile(r"[A-Za-z][A-Za-z'\-]*")

def _syllables(word):
    w = word.lower()
    groups = re.findall(r"[aeiouy]+", w)
    n = len(groups)
    if w.endswith("e") and not w.endswith(("le", "ee")) and n > 1:
        n -= 1
    return max(1, n)

def depth_readout(answer_text):
    out = parse_answer(answer_text) or {}
    briefing = str(out.get("briefing") or "")
    words = _WORD_RE.findall(briefing)
    sentences = [s for s in re.split(r"(?<=[.!?])\s+", briefing.strip()) if s.strip()]
    n_w, n_s = max(1, len(words)), max(1, len(sentences))
    syl = sum(_syllables(w) for w in words)
    grade = 0.39 * (n_w / n_s) + 11.8 * (syl / n_w) - 15.59   # Flesch-Kincaid grade level
    terms = [t for t in out.get("terms_used") or [] if isinstance(t, str)]
    explained = [t for t in out.get("explained_terms") or [] if isinstance(t, str)]
    return {
        "words": len(words),
        "grade_level": round(grade, 1),
        "mean_sentence_len": round(n_w / n_s, 1),
        "domain_terms_per_100w": round(100 * len(terms) / n_w, 1),
        "definitions_per_100w": round(100 * len(explained) / n_w, 1),
    }

def packet_key_facts(row):
    ctx = json.loads(row["messages"][1]["content"])["context"]
    meta = row["meta"]
    event = (ctx.get("events") or [{}])[0]
    known = sorted({t for s in KNOWN_STATES for t in (ctx.get("concept_ledger") or {}).get(s, [])})
    fam = {k: v.get("band") for k, v in (ctx.get("subdomain_familiarity") or {}).items() if v.get("band") != "beginner"}
    lines = [
        f"profile: {meta.get('profile')}   group: {ctx.get('group', {}).get('name')}   band: {ctx.get('proficiency')}",
        f"already knows ({len(known)}): {', '.join(known[:12])}{' ...' if len(known) > 12 else ''}" if known else "already knows: nothing yet",
        f"stronger subdomains: {fam}" if fam else "stronger subdomains: none",
        f"event: {event.get('headline')}",
        "        " + textwrap.shorten(str(event.get("detail") or ""), 260),
    ]
    return ctx, "\n".join(lines)

# ---- generate with the trained model ----
def pick(profile):
    for r in val_rows:
        if r["meta"].get("profile") == profile:
            return r
    return val_rows[0]

def generate_briefing(row, max_new_tokens=768):
    prompt = tokenizer.apply_chat_template(row["messages"][:2], tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=True, temperature=0.3, top_p=0.9,
                             pad_token_id=tokenizer.pad_token_id)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

if USE_UNSLOTH:
    FastLanguageModel.for_inference(model)
else:
    model.eval()

def show(label, text):
    parsed = parse_answer(text) or {}
    print(f"--- {label} ---")
    print(textwrap.fill(str(parsed.get("briefing") or text), 100))
    print(f"    topic: {parsed.get('topic')!r}   explained: {parsed.get('explained_terms')}")

def show_checks(label, checks):
    marks = "  ".join(f"{k}={'ok' if v else 'FAIL'}" for k, v in checks.items() if k not in ("passed", "details"))
    print(f"{label:<8} {'PASS' if checks['passed'] else 'FAIL'}   {marks}")
    if not checks["passed"]:
        print(" " * 9, {k: v for k, v in checks["details"].items() if v})

for profile in ("beginner", "expert_one_subdomain"):
    row = pick(profile)
    ctx, facts = packet_key_facts(row)
    print("=" * 100)
    print(facts)
    print()
    student = generate_briefing(row)
    teacher = row["messages"][2]["content"]
    show("trained model", student)
    print()
    show("Claude (teacher)", teacher)
    print()
    print("hard rules:")
    show_checks("model", hard_rule_checks(ctx, student))
    show_checks("Claude", hard_rule_checks(ctx, teacher))
    print("depth:   model ", depth_readout(student))
    print("         Claude", depth_readout(teacher))
    print()

## 10. Save the model in three forms

Three files come out of this cell, in this order:

1. **The adapter** (about 120 MB): just the LoRA part we trained. Small, quick, and enough to recreate everything else later. If the loss plot showed memorising after epoch 2, set `ADAPTER_FROM_CHECKPOINT` to that epoch's folder name in your working folder (for example `checkpoint-102`) and the cell loads it before saving.
2. **The merged model** (about 6 GB): the adapter folded back into the original weights, saved in full precision. Useful if you later want to train again or serve with other tools; skip it with `SAVE_MERGED_16BIT = False` if you are short on time.
3. **The GGUF file** (about 2 GB): **GGUF** is the file format the local server on your Mac (Ollama) reads. `q4_k_m` means the numbers are stored in about 4 bits each, which is what makes it small enough for an 8 GB laptop. This conversion compiles a small tool the first time and takes 10 to 15 minutes; a lot of text scrolls past, which is normal.

The two big intermediate files stay on Colab's own disk; only the adapter, the checkpoints and the final 2 GB GGUF go to your working folder, so a free 15 GB Drive has room (about 2.5 GB used in total). The last part hands you the GGUF: with `SEND_TO = "drive"` it is in `MyDrive/news-briefer/` and you download it from drive.google.com; with `SEND_TO = "download"` the browser downloads it directly (2 GB, so a few minutes, and keep the tab open).

In [ ]:
import glob, os, shutil

ADAPTER_FROM_CHECKPOINT = None      # e.g. "checkpoint-102" to prefer an earlier epoch
SAVE_MERGED_16BIT = True
SEND_TO = "drive" if USE_DRIVE else "download"

ADAPTER_DIR = os.path.join(WORK_DIR, "news-briefer-lora")     # small: stays with your checkpoints (Drive if connected)
MERGED_DIR = "/content/news-briefer-merged"                    # big (6 GB): Colab's local disk, not Drive
GGUF_DIR = "/content/news-briefer-gguf"                        # big while converting: local disk; the final 2 GB file is copied out
FINAL_GGUF = os.path.join(WORK_DIR, "news-briefer-q4_k_m.gguf")

if ADAPTER_FROM_CHECKPOINT:
    from peft import set_peft_model_state_dict
    from safetensors.torch import load_file
    ckpt = os.path.join(OUTPUT_DIR, ADAPTER_FROM_CHECKPOINT)
    print("loading adapter weights from", ckpt)
    set_peft_model_state_dict(model, load_file(os.path.join(ckpt, "adapter_model.safetensors")))

# 1. adapter
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print("adapter saved to", ADAPTER_DIR)

# 2. merged full model
if SAVE_MERGED_16BIT:
    if USE_UNSLOTH:
        model.save_pretrained_merged(MERGED_DIR, tokenizer, save_method="merged_16bit")
    else:
        merged = model.merge_and_unload()
        merged.save_pretrained(MERGED_DIR, safe_serialization=True)
        tokenizer.save_pretrained(MERGED_DIR)
    print("merged model saved to", MERGED_DIR)

# 3. GGUF (q4_k_m)
if USE_UNSLOTH:
    model.save_pretrained_gguf(GGUF_DIR, tokenizer, quantization_method="q4_k_m")
else:
    import subprocess, sys
    if not SAVE_MERGED_16BIT:
        raise SystemExit("the fallback GGUF path needs the merged model: set SAVE_MERGED_16BIT = True and re-run")
    if not os.path.isdir("/content/llama.cpp"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/ggml-org/llama.cpp", "/content/llama.cpp"], check=True)
        subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-r", "/content/llama.cpp/requirements/requirements-convert_hf_to_gguf.txt"], check=True)
        subprocess.run(["cmake", "-B", "/content/llama.cpp/build", "-S", "/content/llama.cpp", "-DGGML_CUDA=OFF"], check=True)
        subprocess.run(["cmake", "--build", "/content/llama.cpp/build", "--target", "llama-quantize", "-j"], check=True)
    os.makedirs(GGUF_DIR, exist_ok=True)
    f16 = os.path.join(GGUF_DIR, "news-briefer-f16.gguf")
    subprocess.run([sys.executable, "/content/llama.cpp/convert_hf_to_gguf.py", MERGED_DIR, "--outfile", f16, "--outtype", "f16"], check=True)
    subprocess.run(["/content/llama.cpp/build/bin/llama-quantize", f16, os.path.join(GGUF_DIR, "news-briefer-q4_k_m.gguf"), "q4_k_m"], check=True)
    os.remove(f16)

candidates = sorted(glob.glob(os.path.join(GGUF_DIR, "*.gguf")), key=os.path.getsize)
q4 = [p for p in candidates if "q4_k_m" in os.path.basename(p).lower()] or candidates
if not q4:
    raise SystemExit(f"no .gguf file found in {GGUF_DIR}")
print(f"GGUF built: {q4[0]}  ({os.path.getsize(q4[0])/1e9:.2f} GB)")
if os.path.abspath(q4[0]) != os.path.abspath(FINAL_GGUF):
    print("copying it to", FINAL_GGUF, "(a minute or two if that is on Drive)")
    shutil.copyfile(q4[0], FINAL_GGUF)
print(f"GGUF ready: {FINAL_GGUF}  ({os.path.getsize(FINAL_GGUF)/1e9:.2f} GB)")

if SEND_TO == "download":
    from google.colab import files
    files.download(FINAL_GGUF)
else:
    print("It is in your Google Drive under MyDrive/news-briefer/ -- download it from drive.google.com onto your Mac.")

## 11. Run it on your Mac and put it in the briefing seat

Back on your Mac, with the GGUF file downloaded (say to `~/Downloads/news-briefer-q4_k_m.gguf`) and [Ollama](https://ollama.com) installed:

**Before you start: memory.** Your Mac has 8 GB. The model needs about 2.5 GB while running, plus room for the long briefing prompt. Quit the browser and anything else heavy first, or it will crawl or fail to load.

1. The cell below writes a file called `Modelfile` into the working folder and prints it. Download it next to the GGUF (or just copy the printed text into a file called `Modelfile` in the same folder as the GGUF). It tells Ollama three things: which file to load, the chat layout the model was trained with (the `<|im_start|>` markers you saw earlier, exactly the same), and that no system prompt is baked in, because the news agent sends its own on every call.
2. In a terminal, in that folder:
   ```bash
   ollama create news-briefer -f Modelfile
   ollama run news-briefer "Say hello in one sentence."      # a smoke test; Ctrl-D to leave
   ```
3. Point the news agent at it with the two switches, then run the cheap three-persona regression from `training/README.md` (this still needs Claude credentials for every call that is not the briefing):
   ```bash
   export LOCAL_MODEL_BASE_URL=http://localhost:11434/v1
   export LOCAL_MODEL_NAME=news-briefer
   cd /path/to/news_agent
   for p in sam_beginner pilar_wine dana_startup; do
     PYTHONPATH=src:. .venv/bin/python -m harness.run --live --cheap --rounds 4 \
         --persona $p --db eval-runs/local/$p.db --json eval-runs/local/$p.json \
         --materiality-cache eval-runs/cache/materiality.json
   done
   ```
   `cli models` shows `news-briefer` in the briefing row while the switches are set; unset them and everything goes back to Claude.

In [ ]:
import os

TQ = '"' * 3  # Ollama wraps multi-line templates in triple quotes
MODELFILE = "\n".join([
    "FROM ./news-briefer-q4_k_m.gguf",
    "",
    "# The chat layout the model was trained with (Qwen's ChatML). No SYSTEM line on purpose:",
    "# the news agent sends the full briefing prompt as the system message on every call.",
    "TEMPLATE " + TQ + "{{- range .Messages }}<|im_start|>{{ .Role }}",
    "{{ .Content }}<|im_end|>",
    "{{ end }}<|im_start|>assistant",
    TQ,
    "PARAMETER stop <|im_start|>",
    "PARAMETER stop <|im_end|>",
    "PARAMETER num_ctx 8192",
    "PARAMETER temperature 0.3",
    "",
])

path = os.path.join(os.path.dirname(FINAL_GGUF), "Modelfile")  # next to the final GGUF
with open(path, "w", encoding="utf-8") as fh:
    fh.write(MODELFILE)
print("wrote", path)
print()
print(MODELFILE)
print("On your Mac, in the folder holding the GGUF and this Modelfile:")
print("  ollama create news-briefer -f Modelfile")
print("  export LOCAL_MODEL_BASE_URL=http://localhost:11434/v1")
print("  export LOCAL_MODEL_NAME=news-briefer")